# Finetuning DIRE classifier to current dataset

**Pipeline:** Compute DIRE maps for all the dataset → finetune lsun classifier →  Run finetuned classifier → Metrics

# Prepasations

In [ ]:
from google.colab import drive
import zipfile, os, shutil, random, subprocess
import pandas as pd
from PIL import Image

import numpy as np, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (accuracy_score, roc_auc_score,recall_score ,
                             f1_score, confusion_matrix,
                             classification_report)

Mounting drive to get the dataset and the existing DIRE test zip that was computed for the zeroshot

In [ ]:
drive.mount('/content/drive')

# Extracting the dire_test maps already computed
os.makedirs('/content/dire_maps/test', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/Masters/advanced img/dire_test.zip', 'r') as z:
    z.extractall('/content/dire_maps/test')

print('Test DIRE maps extracted.')
print('real:', len(os.listdir('/content/dire_maps/test/content/dire_test/real')))
print('fake:', len(os.listdir('/content/dire_maps/test/content/dire_test/fake')))

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Masters/advanced img/dire_test.zip'

#### Extracting Dataset
#### Spliting into Train / Val / Test (80 / 10 / 10)


In [ ]:
random.seed(42)

# Extract full dataset
os.makedirs('/content/dataset', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/Masters/advanced img/openfake_subset.zip', 'r') as z:
    z.extractall('/content/dataset')

METADATA_PATH = '/content/dataset/metadata.csv'
OUT_BASE      = '/content/splits'

df = pd.read_csv(METADATA_PATH)
df['image_path'] = df['image_path'].str.replace(
    '/content/openfake_subset/images',
    '/content/dataset/images',
    regex=False
)
df['split_label'] = df['label'].apply(
    lambda x: 'real' if str(x).lower() == 'real' else 'fake'
)

def split_df(dataframe, splits=(0.8, 0.1, 0.1)):
    dataframe = dataframe.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(dataframe)
    n_train = int(n * splits[0])
    n_val   = int(n * splits[1])
    return {
        'train': dataframe.iloc[:n_train],
        'val'  : dataframe.iloc[n_train:n_train+n_val],
        'test' : dataframe.iloc[n_train+n_val:]
    }

def copy_split(split_dict, label, out_base):
    for split_name, sdf in split_dict.items():
        dst = os.path.join(out_base, split_name, label)
        os.makedirs(dst, exist_ok=True)
        copied, missing = 0, 0
        for _, row in sdf.iterrows():
            src = row['image_path']
            if not os.path.exists(src):
                missing += 1
            else:
                shutil.copy(src, os.path.join(dst, os.path.basename(src)))
                copied += 1
        print(f'  {split_name}/{label}: {copied} copied, {missing} missing')

real_df = df[df['split_label'] == 'real'].copy()
fake_df = df[df['split_label'] == 'fake'].copy()

copy_split(split_df(real_df), 'real', OUT_BASE)
copy_split(split_df(fake_df), 'fake', OUT_BASE)
print('Splits ready.')

  train/real: 2400 copied, 0 missing
  val/real: 300 copied, 0 missing
  test/real: 300 copied, 0 missing
  train/fake: 4791 copied, 0 missing
  val/fake: 598 copied, 0 missing
  test/fake: 600 copied, 0 missing
Splits ready.


#### Resizing All Images to 256×256 (the ADM model requires this size for the images)


In [ ]:
def resize_folder(folder):
    count = 0
    for root, _, files in os.walk(folder):
        for f in files:
            if not f.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
            path = os.path.join(root, f)
            img  = Image.open(path).convert('RGB')
            if img.size != (256, 256):
                img = img.resize((256, 256), Image.LANCZOS)
            new_path = os.path.splitext(path)[0] + '.png'
            img.save(new_path, 'PNG')
            if new_path != path:
                os.remove(path)
            count += 1
    print(f'  {count} images resized in {folder}')

# here we only resize train and val -> test DIRE maps are already done
for split in ['train', 'val']:
    for label in ['real', 'fake']:
        resize_folder(f'/content/splits/{split}/{label}')
print('Done.')

  2400 images resized in /content/splits/train/real
  4791 images resized in /content/splits/train/fake
  300 images resized in /content/splits/val/real
  598 images resized in /content/splits/val/fake
Done.


# Getting the pre trained models

#### Cloning DIRE reopository to get the compute_dire.py file to pass the test data to the file directly


In [ ]:
%%bash
cd /content
git clone https://github.com/ZhendongWang6/DIRE.git
cd DIRE
pip install -q -r requirements.txt
# mpi4py is needed for compute_dire.py
pip install -q mpi4py
echo 'DIRE repo ready.'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.2 MB/s eta 0:00:00
DIRE repo ready.


Cloning into 'DIRE'...


#### Getting the ADM diffusion model (for reconstruction / DIRE computation)
This model is publicly available at OpenAI

In [ ]:
%%bash
mkdir -p /content/DIRE/models
cd /content/DIRE/models

# Downloading the ADM 256x256 unconditional diffusion model (OpenAI public mirror)
wget -q --show-progress \
  https://openaipublic.blob.core.windows.net/diffusion/jul-2021/256x256_diffusion_uncond.pt \
  -O 256x256_diffusion_uncond.pt

echo 'ADM diffusion model downloaded.'
ls -lh /content/DIRE/models/

ADM diffusion model downloaded.
total 2.1G
-rw-r--r-- 1 root root 2.1G Jul 19  2021 256x256_diffusion_uncond.pt



     0K .......... .......... .......... .......... ..........  0%  126K 4h46m
    50K .......... .......... .......... .......... ..........  0%  251K 3h35m
   100K .......... .......... .......... .......... ..........  0%  212M 2h23m
   150K .......... .......... .......... .......... ..........  0%  251K 2h23m
   200K .......... .......... .......... .......... ..........  0%  233M 1h54m
   250K .......... .......... .......... .......... ..........  0%  251K 1h59m
   300K .......... .......... .......... .......... ..........  0%  245M 1h42m
   350K .......... .......... .......... .......... ..........  0%  289M 89m40s
   400K .......... .......... .......... .......... ..........  0%  251K 95m36s
   450K .......... .......... .......... .......... ..........  0%  162M 86m4s
   500K .......... .......... .......... .......... ..........  0%  312M 78m15s
   550K .......... .......... .......... .......... ..........  0%  256M 71m44s
   600K .......... .......... .......... ......

#### Getting the pretrained ResNet-50 classifier (lsun_adm.pth)
This model is available in the DIRE repository README.
I downloaded it and upload it to my drive to be reused through multiple sessions and files

In [ ]:
import shutil
import os

os.makedirs('/content/DIRE/models', exist_ok=True)
shutil.copy(
    '/content/drive/MyDrive/Masters/advanced img/lsun_adm.pth',
    '/content/DIRE/models/lsun_adm.pth'
)
print('Classifier weights copied.')

Classifier weights copied.


# Computing DIRE Representations for the training and validation Sets

Since this cannot be done in one session, the code was adjusted to be resumable, every map that is done is saved directly to my drive so that when the session is opened again the dire generation can resume exactly where it left off  it skips images already processed

In [ ]:
# Creating symlinks first
DRIVE_DIRE ='/content/drive/MyDrive/Masters/advanced img/dire_maps'
DRIVE_RECON = '/content/drive/MyDrive/Masters/advanced img/recon_maps'

os.makedirs(DRIVE_DIRE,  exist_ok=True)
os.makedirs(DRIVE_RECON, exist_ok=True)

for link in ['/content/dire_drive', '/content/recon_drive']:
    if os.path.islink(link):
        os.unlink(link)

os.symlink(DRIVE_DIRE,'/content/dire_drive')
os.symlink(DRIVE_RECON,'/content/recon_drive')

print('dire_drive  →', os.readlink('/content/dire_drive'))
print('recon_drive →', os.readlink('/content/recon_drive'))
print('Symlinks working:', os.path.exists('/content/dire_drive'))

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

MODEL_PATH  = '/content/DIRE/models/256x256_diffusion_uncond.pt'
MODEL_FLAGS = (
    '--attention_resolutions 32,16,8 '
    '--class_cond False '
    '--diffusion_steps 1000 '
    '--dropout 0.1 '
    '--image_size 256 '
    '--learn_sigma True '
    '--noise_schedule linear '
    '--num_channels 256 '
    '--num_head_channels 64 '
    '--num_res_blocks 2 '
    '--resblock_updown True '
    '--use_fp16 True '
    '--use_scale_shift_norm True'
)

DRIVE_DIRE_BASE  = '/content/dire_drive'
DRIVE_RECON_BASE = '/content/recon_drive'

def get_stems(folder, exts=('.png', '.jpg', '.jpeg')):
    """Filenames without extension from a folder."""
    if not os.path.exists(folder):
        return set()
    return {
        os.path.splitext(f)[0]
        for f in os.listdir(folder)
        if f.lower().endswith(exts)
    }

def run_compute_dire(images_dir, recons_dir, dire_dir):
    os.makedirs(dire_dir,  exist_ok=True)
    os.makedirs(recons_dir, exist_ok=True)

    all_stems = get_stems(images_dir)
    done_stems = get_stems(dire_dir)
    todo_stems = all_stems - done_stems

    total = len(all_stems)
    already = len(done_stems)
    remaining = len(todo_stems)

    print(f'  Total: {total} | Already on Drive: {already} | Remaining: {remaining}')

    if remaining == 0:
        print(f'  Already complete — skipping.')
        return

    # Copy only unprocessed images to a clean temp folder
    TEMP_IN = '/content/dire_temp_in'
    TEMP_RECON = '/content/dire_temp_recon'
    shutil.rmtree(TEMP_IN,    ignore_errors=True)
    shutil.rmtree(TEMP_RECON, ignore_errors=True)
    os.makedirs(TEMP_IN,    exist_ok=True)
    os.makedirs(TEMP_RECON, exist_ok=True)

    for stem in todo_stems:
        for ext in ['.png', '.jpg', '.jpeg']:
            src = os.path.join(images_dir, stem + ext)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(TEMP_IN, stem + ext))
                break

    copied = len(os.listdir(TEMP_IN))
    print(f'  Copied {copied} unprocessed images to temp folder.')

    # Run compute_dire.py — output goes directly to Drive via symlink
    cmd = (
        f'cd /content/DIRE/guided-diffusion/ && '
        f'CUDA_VISIBLE_DEVICES=0 '
        f'mpiexec --allow-run-as-root -n 1 python compute_dire.py '
        f'--model_path {MODEL_PATH} '
        f'{MODEL_FLAGS} '
        f'--images_dir {TEMP_IN} '
        f'--recons_dir {TEMP_RECON} '
        f'--dire_dir {dire_dir} '
        f'--batch_size 1 '
        f'--num_samples {copied} '
        f'--timestep_respacing ddim20 '
        f'--use_ddim True '
        f'--has_subfolder False'
    )

    process = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in process.stdout:
        print(line, end='')
    process.wait()

    # Clean up temp folders
    shutil.rmtree(TEMP_IN,    ignore_errors=True)
    shutil.rmtree(TEMP_RECON, ignore_errors=True)

    done_now = len(get_stems(dire_dir))
    print(f'  Session done: {done_now}/{total} DIRE maps on Drive.')
    print(f'  Remaining after this session: {total - done_now}')

for split in ['train', 'val']:
    for label in ['real', 'fake']:
        print(f'\n=== {split}/{label} ===')
        run_compute_dire(
            images_dir = f'/content/splits/{split}/{label}',
            recons_dir = f'{DRIVE_RECON_BASE}/{split}/{label}',
            dire_dir = f'{DRIVE_DIRE_BASE}/{split}/{label}'
        )

print('\nAll done. DIRE maps saved at:', DRIVE_DIRE)

dire_drive  → /content/drive/MyDrive/Masters/advanced img/dire_maps
recon_drive → /content/drive/MyDrive/Masters/advanced img/recon_maps
Symlinks working: True

=== train/real ===
  Total: 2400 | Already on Drive: 2400 | Remaining: 0
  Already complete — skipping.

=== train/fake ===
  Total: 4791 | Already on Drive: 4791 | Remaining: 0
  Already complete — skipping.

=== val/real ===
  Total: 300 | Already on Drive: 300 | Remaining: 0
  Already complete — skipping.

=== val/fake ===
  Total: 598 | Already on Drive: 598 | Remaining: 0
  Already complete — skipping.

All done. DIRE maps saved at: /content/drive/MyDrive/Masters/advanced img/dire_maps


# Fine-tuning the ResNet-50 classifier


In [ ]:
# Getting the dire maps from the drive
DRIVE_DIRE = '/content/drive/MyDrive/Masters/advanced img/dire_maps'
for link, target in [('/content/dire_drive', DRIVE_DIRE)]:
    if os.path.islink(link):
        os.unlink(link)
    os.symlink(target, link)

def prepare_imagefolder(split):
    for label, folder_name in [('real', '0_real'), ('fake', '1_fake')]:
        src = f'/content/dire_drive/{split}/{label}'
        dst = f'/content/imagefolder/{split}/{folder_name}'
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.islink(dst):
            os.unlink(dst)
        if not os.path.exists(dst):
            os.symlink(src, dst)

prepare_imagefolder('train')
prepare_imagefolder('val')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/content/imagefolder/train', transform=transform)
val_ds   = datasets.ImageFolder('/content/imagefolder/val',   transform=transform)

# Weighted sampler to handle 1:2 real:fake imbalance
counts  = [train_ds.targets.count(i) for i in range(2)]
weights = [1.0 / counts[t] for t in train_ds.targets]
sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

# Setting batch size to 8 is a decision made for all the models that are being finetuned in this comparitive analysis
train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2)
val_loader= DataLoader(val_ds,   batch_size=8, shuffle=False,   num_workers=2)

print(f'Train: {len(train_ds)} images  |  Val: {len(val_ds)} images')
print(f'Classes: {train_ds.classes}')

# Loading  pretrained classifier
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.resnet50(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)

ckpt  = torch.load('/content/DIRE/models/lsun_adm.pth', map_location=device)
state = ckpt if isinstance(ckpt, dict) and 'state_dict' not in ckpt else ckpt.get('state_dict', ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
print(f'Loaded weights — missing: {len(missing)}, unexpected: {len(unexpected)}')

# freezing all parameters first, then unfreeze ~20% of the parameters ( this percentage is also set for all models)
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last bottleneck of layer4 + FC head ≈ 20% of parameters
for param in model.layer4[-1].parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable_params:,} / {total_params:,} '
      f'({100 * trainable_params / total_params:.1f}%)')

model = model.to(device)

criterion = nn.CrossEntropyLoss()
#  only passing trainable parameters to optimizer
optimizer = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=10)

#Number of epochs is agreed upon for all models in the comparative analsys as well
EPOCHS = 10
SAVE_PATH = '/content/DIRE/models/dire_finetuned.pth'
DRIVE_SAVE= '/content/drive/MyDrive/Masters/advanced img/dire_finetuned.pth'
best_val_auc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
            val_probs.extend(probs)
            val_true.extend(labels.numpy())

    val_auc = roc_auc_score(val_true, val_probs)
    val_acc = np.mean(np.array(val_probs) >= 0.5)
    scheduler.step()

    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train loss: {train_loss/len(train_loader):.4f} | '
          f'Train acc: {correct/total:.4f} | '
          f'Val AUC: {val_auc:.4f}')

    # Saving best model to Drive
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)
        shutil.copy(SAVE_PATH, DRIVE_SAVE)
        print(f'  → Best model saved (AUC: {val_auc:.4f})')

print(f'\nTraining complete. Best Val AUC: {best_val_auc:.4f}')

Train: 7191 images  |  Val: 898 images
Classes: ['0_real', '1_fake']
Device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded weights — missing: 267, unexpected: 3
Trainable: 4,466,690 / 23,512,130 (19.0%)
Epoch 01/10 | Train loss: 0.7011 | Train acc: 0.5535 | Val AUC: 0.6934
  → Best model saved (AUC: 0.6934)
Epoch 02/10 | Train loss: 0.6656 | Train acc: 0.5948 | Val AUC: 0.6270
Epoch 03/10 | Train loss: 0.6507 | Train acc: 0.6261 | Val AUC: 0.7042
  → Best model saved (AUC: 0.7042)
Epoch 04/10 | Train loss: 0.6426 | Train acc: 0.6243 | Val AUC: 0.7046
  → Best model saved (AUC: 0.7046)
Epoch 05/10 | Train loss: 0.6245 | Train acc: 0.6430 | Val AUC: 0.7045
Epoch 06/10 | Train loss: 0.6265 | Train acc: 0.6415 | Val AUC: 0.7220
  → Best model saved (AUC: 0.7220)
Epoch 07/10 | Train loss: 0.6163 | Train acc: 0.6526 | Val AUC: 0.6987
Epoch 08/10 | Train loss: 0.6097 | Train acc: 0.6539 | Val AUC: 0.7081
Epoch 09/10 | Train loss: 0.6070 | Train acc: 0.6466 | Val AUC: 0.7043
Epoch 10/10 | Train loss: 0.6003 | Train acc: 0.6583 | Val AUC: 0.7043

Training complete. Best Val AUC: 0.7220


# Evaluating fine-tuned model on the test DIRE maps


In [ ]:
DRIVE_DIRE = '/content/drive/MyDrive/Masters/advanced img/dire_maps'
if os.path.islink('/content/dire_drive'):
    os.unlink('/content/dire_drive')
os.symlink(DRIVE_DIRE, '/content/dire_drive')

# The test DIRE maps were extracted to /content/dire_maps/test/content/dire_test/
TEST_DIRE_MAPS_PATH = '/content/dire_maps/test/content/dire_test'

for label, folder_name in [('real', '0_real'), ('fake', '1_fake')]:
    src = f'{TEST_DIRE_MAPS_PATH}/{label}' # Corrected source path
    dst = f'/content/imagefolder/test/{folder_name}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    if not os.path.exists(dst):
        os.symlink(src, dst)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_ds     = datasets.ImageFolder('/content/imagefolder/test', transform=transform)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)
print(f'Test images : {len(test_ds)}')
print(f'Classes: {test_ds.classes}')   # ['0_real', '1_fake']

# Loading fine-tuned model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

# Rebuilding the same frozen/unfrozen structure as when we were finetuning
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4[-1].parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model.load_state_dict(torch.load(
    '/content/drive/MyDrive/Masters/advanced img/dire_finetuned.pth',
    map_location=device
))
model = model.to(device)
model.eval()

all_probs, all_true, all_fnames = [], [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_true.extend(labels.numpy())

y_true = np.array(all_true)
y_prob = np.array(all_probs)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
f1 = f1_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

# Printing results
print('=' * 40)
print('  DIRE Fine-Tuned Results')
print('=' * 40)
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  AUC      : {auc:.4f}   (zero-shot was 0.5174)')
print(f'  F1 Score : {f1:.4f}   (zero-shot was 0.7997)')
print(f'  Recall   : {recall:.4f}   (zero-shot was 0.998)')
print('=' * 40)
print(f'\nConfusion Matrix (rows=true, cols=pred):')
print(f'               Pred Real  Pred Fake')
print(f'  True Real  :   {cm[0][0]:>5}      {cm[0][1]:>5}')
print(f'  True Fake  :   {cm[1][0]:>5}      {cm[1][1]:>5}')
print(f'\nFull Report:')
print(classification_report(y_true, y_pred, target_names=['real', 'fake']))

#Saving results JSON to Drive for comparison
results = {
    'model': 'dire_finetuned.pth (fine-tuned)',
    'accuracy' : round(float(acc), 4),
    'auc': round(float(auc), 4),
    'f1': round(float(f1),  4),
    'recall': round(float(recall),4),
    'n_real': int((y_true == 0).sum()),
    'n_fake' : int((y_true == 1).sum()),
    'confusion_matrix': {
        'true_real_pred_real' : int(cm[0][0]),
        'true_real_pred_fake' : int(cm[0][1]),
        'true_fake_pred_real' : int(cm[1][0]),
        'true_fake_pred_fake' : int(cm[1][1])
    },
    'zero_shot_comparison': {
        'auc_zeroshot'   : 0.5174,
        'auc_finetuned'  : round(float(auc), 4),
        'auc_improvement': round(float(auc) - 0.5174, 4)
    },
    'note': 'Fine-tuned on OpenFake DIRE maps. 19% trainable params (layer4[-1] + FC). Batch size 8. 10 epochs.'
}

DRIVE_OUT = '/content/drive/MyDrive/Masters/advanced img/dire_finetuned_results.json'
with open(DRIVE_OUT, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults JSON saved to Drive.')
print(json.dumps(results, indent=2))

# Saving per-image CSV to Drive
fnames = [os.path.basename(p) for p, _ in test_ds.samples]
pd.DataFrame({
    'filename'  : fnames,
    'true'      : y_true,
    'true_label': ['real' if t == 0 else 'fake' for t in y_true],
    'prob_fake' : y_prob,
    'pred'      : y_pred,
    'correct'   : (y_pred == y_true).astype(int)
}).to_csv(
    '/content/drive/MyDrive/Masters/advanced img/dire_finetuned_predictions.csv',
    index=False
)
print('Per-image CSV saved to Drive.')

Test images : 900
Classes: ['0_real', '1_fake']
Device: cuda
  DIRE Fine-Tuned Results
  Accuracy : 0.6811  (68.11%)
  AUC      : 0.7063   (zero-shot was 0.5174)
  F1 Score : 0.7570   (zero-shot was 0.7997)
  Recall   : 0.7450   (zero-shot was 0.998)

Confusion Matrix (rows=true, cols=pred):
               Pred Real  Pred Fake
  True Real  :     166        134
  True Fake  :     153        447

Full Report:
              precision    recall  f1-score   support

        real       0.52      0.55      0.54       300
        fake       0.77      0.74      0.76       600

    accuracy                           0.68       900
   macro avg       0.64      0.65      0.65       900
weighted avg       0.69      0.68      0.68       900


Results JSON saved to Drive.
{
  "model": "dire_finetuned.pth (fine-tuned)",
  "accuracy": 0.6811,
  "auc": 0.7063,
  "f1": 0.757,
  "recall": 0.745,
  "n_real": 300,
  "n_fake": 600,
  "confusion_matrix": {
    "true_real_pred_real": 166,
    "true_real_pred_f

The AUC is the honest number, and it improved massively

```
Zero-shot AUC  : 0.517  ←  random
Fine-tuned AUC : 0.706  ←  genuinely learning
Improvement    : +0.189
```

**Accuracy barely moved (66.67% → 68.11%):**

The zero-shot got 66.67% by predicting everything as fake. the fine-tuned model gets 68.11% but in a completely different way
 The fine-tuned model now correctly identifies 166/300 real images. That is a massive behavioral change that accuracy alone doesn't capture because of the 1:2 class imbalance.

**F1 went down (0.799 → 0.757):**

The zero-shot F1 was artificially inflated. F1 is calculated on the fake class only by default, and since the zero-shot predicted everything as fake it got near-perfect fake recall (599/600). The fine-tuned model is more conservative, it doesn't just call everything fake anymore, so the F1 on fake dropped slightly, but that's actually a sign of a more honest and balanced model.

# Evaluating the performance per difusion generator





In [ ]:
ZIP_PATH = '/content/drive/MyDrive/Masters/advanced img/openfake_subset.zip'
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}. Check the path.'
print(f'Found zip: {ZIP_PATH}')
EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting zip (this may take a few minutes)...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Done.')

Found zip: /content/drive/MyDrive/Masters/advanced img/openfake_subset.zip
Extracting zip (this may take a few minutes)...
Done.


In [ ]:
test_fnames = [os.path.basename(p) for p, _ in test_ds.samples]

results_df = pd.DataFrame({
    'filename' : test_fnames,
    'true'     : y_true,
    'prob_fake': y_prob,
    'pred'     : y_pred
})

results_df['stem'] = results_df['filename'].apply(lambda f: os.path.splitext(f)[0])

meta = pd.read_csv('/content/dataset/metadata.csv')
meta['stem'] = meta['image_path'].apply(lambda p: os.path.splitext(os.path.basename(p))[0])

results_df = results_df.merge(meta[['stem', 'label', 'model']], on='stem', how='left')

print(f"Matched: {results_df['label'].notna().sum()} / {len(results_df)}")
print("Label distribution:", results_df['label'].value_counts().to_dict())

print('\n=== Per-Generator Performance (Fine-Tuned DIRE) ===')
real_subset = results_df[results_df['label'] == 'real']

for gen_label in ['stable_diffusion', 'midjourney']:
    subset = results_df[results_df['label'] == gen_label]
    if len(subset) == 0:
        continue
    combined = pd.concat([real_subset, subset])
    acc_g    = accuracy_score(combined['true'], combined['pred'])
    try:
        auc_g = roc_auc_score(combined['true'], combined['prob_fake'])
    except:
        auc_g = float('nan')
    f1_g     = f1_score(combined['true'], combined['pred'], zero_division=0)
    recall_g = recall_score(combined['true'], combined['pred'], zero_division=0)
    print(f'\n  {gen_label}  (n={len(subset)} fake + {len(real_subset)} real)')
    print(f'    Accuracy : {acc_g:.4f}    AUC : {auc_g:.4f}    F1 : {f1_g:.4f}    Recall : {recall_g:.4f}')

print('\n=== SD Sub-Model Breakdown ===')
sd_subset = results_df[results_df['label'] == 'stable_diffusion']
for model_name, grp in sd_subset.groupby('model'):
    fake_recall = grp['pred'].mean()
    print(f'  {model_name:35s}  n={len(grp):4d}  fake_recall={fake_recall:.3f}')

Matched: 900 / 900
Label distribution: {'midjourney': 304, 'real': 300, 'stable_diffusion': 296}

=== Per-Generator Performance (Fine-Tuned DIRE) ===

  stable_diffusion  (n=296 fake + 300 real)
    Accuracy : 0.6208    AUC : 0.6776    F1 : 0.6435    Recall : 0.6892

  midjourney  (n=304 fake + 300 real)
    Accuracy : 0.6772    AUC : 0.7342    F1 : 0.7137    Recall : 0.7993

=== SD Sub-Model Breakdown ===
  sd-1.5                               n=  14  fake_recall=1.000
  sd-1.5-dreamshaper                   n=  21  fake_recall=0.952
  sd-1.5-epicdream                     n=   9  fake_recall=0.667
  sd-2.1                               n=  67  fake_recall=0.761
  sd-3.5                               n=  68  fake_recall=0.368
  sdxl                                 n=  25  fake_recall=0.880
  sdxl-epic-realism                    n=  34  fake_recall=0.765
  sdxl-juggernaut                      n=  16  fake_recall=0.500
  sdxl-realvis-v5                      n=  21  fake_recall=0.667
  sdx

#  Evaluating on Out of distribution test set

Out of distribution test set, 900 fake from 18 different generators and 450 real to keep the 1:2 ratio that we're working with

Dire maps were already created when evaluating the zero shot

In [ ]:
OOD_DIRE_REAL  = '/'
OOD_DIRE_FAKE= '/content/ood_dire/fake'

# Extracting the dire maps already computed
os.makedirs(OOD_DIRE_REAL, exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/Masters/advanced img/ood_dire_real.zip', 'r') as z:
    z.extractall(OOD_DIRE_REAL)

print('Test DIRE maps extracted.')
print('real:', len(os.listdir(/content/ood_dire/real)))

shutil.copytree('/content/drive/MyDrive/Masters/advanced img/ood_dire_fake', OOD_DIRE_FAKE)

print('fake:', len(os.listdir(OOD_DIRE_FAKE)))

Test DIRE maps extracted.
real: 30
fake: 900


In [ ]:
for label, folder_name in [('real', '0_real'), ('fake', '1_fake')]:
    src = f'/content/ood_dire/{label}'
    dst = f'/content/imagefolder/ood/{folder_name}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    if not os.path.exists(dst): os.symlink(src, dst)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

ood_ds     = datasets.ImageFolder('/content/imagefolder/ood', transform=transform)
ood_loader = DataLoader(ood_ds, batch_size=8, shuffle=False, num_workers=2)
print(f'OOD images: {len(ood_ds)}  |  Classes: {ood_ds.classes}')

# Load fine-tuned model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
for param in model.parameters(): param.requires_grad = False
for param in model.layer4[-1].parameters(): param.requires_grad = True
for param in model.fc.parameters(): param.requires_grad = True
model.load_state_dict(torch.load(
    '/content/drive/MyDrive/Masters/advanced img/dire_finetuned.pth',
    map_location=device
))
model = model.to(device)
model.eval()

ood_probs, ood_true = [], []
with torch.no_grad():
    for imgs, labels in ood_loader:
        imgs  = imgs.to(device)
        probs = torch.softmax(model(imgs), dim=1)[:, 1].cpu().numpy()
        ood_probs.extend(probs)
        ood_true.extend(labels.numpy())

ood_true = np.array(ood_true)
ood_prob = np.array(ood_probs)
ood_pred = (ood_prob >= 0.5).astype(int)

OOD images: 1350  |  Classes: ['0_real', '1_fake']


In [ ]:
acc    = accuracy_score(ood_true, ood_pred)
auc    = roc_auc_score(ood_true, ood_prob)
f1     = f1_score(ood_true, ood_pred, zero_division=0)
recall = recall_score(ood_true, ood_pred, zero_division=0)
cm     = confusion_matrix(ood_true, ood_pred)

print('\n' + '=' * 40)
print('  DIRE Fine-Tuned — OOD Results')
print('=' * 40)
print(f'  Accuracy : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  AUC      : {auc:.4f}')
print(f'  F1 Score : {f1:.4f}')
print(f'  Recall   : {recall:.4f}')
print('=' * 40)
print(f'\nConfusion Matrix (rows=true, cols=pred):')
print(f'               Pred Real  Pred Fake')
print(f'  True Real  :   {cm[0][0]:>5}      {cm[0][1]:>5}')
print(f'  True Fake  :   {cm[1][0]:>5}      {cm[1][1]:>5}')

# Saving to Drive ---
ood_results = {
    'model'    : 'dire_finetuned.pth (fine-tuned) — OOD',
    'accuracy' : round(float(acc), 4),
    'auc'      : round(float(auc), 4),
    'f1'       : round(float(f1),  4),
    'recall'   : round(float(recall), 4),
    'n_real'   : int((ood_true == 0).sum()),
    'n_fake'   : int((ood_true == 1).sum()),
    'confusion_matrix': {
        'true_real_pred_real': int(cm[0][0]),
        'true_real_pred_fake': int(cm[0][1]),
        'true_fake_pred_real': int(cm[1][0]),
        'true_fake_pred_fake': int(cm[1][1])
    }
}
with open('/content/drive/MyDrive/Masters/advanced img/dire_finetuned_ood_results.json', 'w') as f:
    json.dump(ood_results, f, indent=2)
print('\nOOD results saved to Drive.')
print(json.dumps(ood_results, indent=2))


  DIRE Fine-Tuned — OOD Results
  Accuracy : 0.4911  (49.11%)
  AUC      : 0.3354
  F1 Score : 0.6280
  Recall   : 0.6444

Confusion Matrix (rows=true, cols=pred):
               Pred Real  Pred Fake
  True Real  :      83        367
  True Fake  :     320        580

OOD results saved to Drive.
{
  "model": "dire_finetuned.pth (fine-tuned) \u2014 OOD",
  "accuracy": 0.4911,
  "auc": 0.3354,
  "f1": 0.628,
  "recall": 0.6444,
  "n_real": 450,
  "n_fake": 900,
  "confusion_matrix": {
    "true_real_pred_real": 83,
    "true_real_pred_fake": 367,
    "true_fake_pred_real": 320,
    "true_fake_pred_fake": 580
  }
}
